# 02 — Baseline: PaddleOCR + Regex / Keyword / Rules
`Image → PaddleOCR → Rules → Entities` (NAME, EMAIL, PHONE, ADDRESS, SECTIONHEADER).

> **Honesty note:** this notebook ships **without outputs**. Every number you see after running it is produced by the code on your machine. Nothing is pre-filled.

In [ ]:
# Run from anywhere inside the repo: move to project root so src/ imports and relative paths work
import os, sys
from pathlib import Path
root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))
print("Project root:", root)

## Live OCR on one test image
PaddleOCR runs on CPU by default (keeps the 4 GB GPU free for LayoutLMv3). Set `RESUMEAI_OCR_GPU=1` to change.

In [ ]:
from src.extraction.training import load_processed
from src.document_processing.ocr import OCRProcessor
from src.extraction.baseline import RuleBasedExtractor
ds, labels = load_processed()
doc = ds["test"][0]
ocr = OCRProcessor()
res = ocr.extract(doc["image"])
print("OCR words:", len(res["words"]))
list(zip(res["words"], res["boxes"], res["scores"]))[:15]

In [ ]:
rules = RuleBasedExtractor()
out = rules.extract(res["words"], res["boxes"])
out

## Rules on the gold (annotated) tokens
This isolates the rule quality from OCR errors.

In [ ]:
pred = rules.tag(doc["words"], doc["boxes"])
[(w, g, p) for w, g, p in zip(doc["words"], doc["labels"], pred) if g != "O" or p != "O"][:40]

## Actual metrics on the full test split
`--mode tokens`: entity-level seqeval on gold tokens (directly comparable with LayoutLMv3).  
`--mode ocr`: PaddleOCR on each image, then string-level matching against gold entities.

In [ ]:
!python scripts/evaluate_baseline.py --mode tokens

In [ ]:
!python scripts/evaluate_baseline.py --mode ocr

In [ ]:
import pandas as pd
pd.read_csv("results/baseline_results.csv")